## problem statement: build a single report that answers:
1.  Top 10 IMDb-rated Netflix titles
2.  Average IMDb rating by content type
3.  Top 10 directors by average rating
4.  Top 10 countries by average rating
5.  Rating trend by release year
6.  Count of Movies vs TV Shows with IMDb Rating > 8
7.  Highest-rated Indian content
8.  Oldest title with Rating > 8
9.  Netflix titles without IMDb Rating
10. IMDb titles not available on Netflix

In [78]:
import pandas as pd

In [ ]:
def load_and_clean_data(netflix_file, ratings_file):
    """loads, clean duplicates and merges the datasets"""
    # load raw data
    netflix = pd.read_csv(netflix_file)
    ratings = pd.read_csv(ratings_file)

    # remove duplicates from ratings
    ratings = ratings.drop_duplicates(subset=["title", "release_year"])

    # merge datasets cleanly
    merged_df = netflix.merge(ratings, on=["title", "release_year"], how="outer", indicator=True)

    return merged_df

clean_df = load_and_clean_data("netflix_titles.csv", "imdb_ratings.csv")
print(clean_df.head().to_string(index=False))

In [ ]:
def run_quality_and_coverage_module(clean_df):
    """Module 1: Validate data coverage and flags missing records between tables."""
    print("\n" + "="*20 + " MODULE 1: QUALITY & COVERAGE " + "="*20)

    # Count Netflix titles without an IMDb Rating
    netflix_missing_count = clean_df.loc[
        (clean_df["show_id"].notnull()) &
        (clean_df["Rating"].isnull())
    ].shape[0]
    print(f"Total Netflix titles missing IMDB Ratings: {netflix_missing_count}")
    print("-" * 70)
    
    # List IMDb titles not available on Netflix (just get the title column)
    imdb_missing_titles = clean_df.loc[
        (clean_df["show_id"].isnull()) &
        (clean_df["Rating"].notnull()),
        ["title"]
    ]
    print("IMDB titles not available on Netflix (Top 5 samples):")
    print(imdb_missing_titles.head().to_string(index=False))
    print(f"\nTotal IMDB titles missing on Netfli: {len(imdb_missing_titles)}")

    print("="*70 + "\n")

run_quality_and_coverage_module(clean_df)

In [ ]:
def run_top_performers_module(clean_df):
    """Module 2: Extracts elite performers, leaderboards, and historical milestones."""
    print("\n" + "="*20 + " MODULE 2: TOP-PERFORMER LEADERBOARDS " + "="*20)

    # Production Safeguard: Filter out IMDB-only records
    netflix_only = clean_df[clean_df["show_id"].notnull()]

    # Top 10 IMDb-rated Netflix titles [ Expected: "title", "Rating" ]
    print("Top 10 IMDb-Rated Netflix Titles:")
    imdb_rated_netflix_title = netflix_only[["title", "Rating"]].sort_values(by="Rating", ascending=False).head(10)
    print(imdb_rated_netflix_title.to_string(index=False))
    print("-" * 78)
    
    # Highest-rated Indian content [ Expected: "title", "director", "country", "Rating" ]
    print("Highest-Rated Indian Content:")
    highest_rated_indian_content = netflix_only.loc[
        netflix_only["country"].str.contains("India", na=False)
    ].sort_values(by="Rating", ascending=False).head(1)[["title", "director", "country", "Rating"]]
    print(highest_rated_indian_content.to_string(index=False))
    print("-" * 78)
    
    # Oldest title with Rating > 8 [ Expected: "title", "release_year" ]
    print("Oldest Title with an IMDb Rating > 8:")
    oldest_title_imdb_rating_more_than_8 = netflix_only[
        netflix_only["Rating"] > 8
    ].sort_values(by="release_year").head(1)[["title", "release_year"]]
    print(oldest_title_imdb_rating_more_than_8.to_string(index=False))

    print("="*78 + "\n")

run_top_performers_module(clean_df)

In [ ]:
def run_grouped_insights_module(clean_df):
    """Module 3: Generates high-level categorical summaries and entity-level rankings."""
    print("\n" + "="*20 + " MODULE 3: GROUPED PERFORMANCE INSIGHTS " + "="*20)

    # Production Safeguard: Filter out IMDB-only records
    netflix_only = clean_df[clean_df["show_id"].notnull()]

    # Average IMDb rating by content type [ Expected: "type", "Avg_Rating" ]
    print("Average IMDb Rating by Content Type:")
    avg_imdb_rating_by_content_type = (
        netflix_only.groupby("type")["Rating"]
        .mean()
        .reset_index()
        .rename(columns={"Rating": "Avg_Rating"})
    )
    print(avg_imdb_rating_by_content_type.to_string(index=False))
    print("-" * 80)
    
    # Top 10 directors by average rating [ Expected: "director", "Avg_Rating" ]
    print("Top 10 Directors by Average IMDb Rating:")
    top_directors_by_avg_imdb_rating = (
        # to avoid miscalculation need to remove null values
        netflix_only.dropna(subset=["director"])        # drop/ignore rows where the director is null before grouping ( dropna() helps to remove null values in one go )
        .groupby("director")["Rating"]
        .mean()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
        .rename(columns={"Rating": "Avg_Rating"})
    )
    print(top_directors_by_avg_imdb_rating.to_string(index=False))
    print("-" * 80)
    
    # Top 10 countries by average rating [ Expected: "country", "Avg_Rating" ]
    print("Top 10 Countries by Average IMDb Rating:")
    top_countries_by_avg_imdb_rating = (
        # to avoid miscalculation need to remove null values
        netflix_only.dropna(subset=["country"])        # drop/ignore rows where the country is null before grouping
        .groupby("country")["Rating"]
        .mean()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
        .rename(columns={"Rating": "Avg_Rating"})
    )
    print(top_countries_by_avg_imdb_rating.to_string(index=False))
    
    print("="*80 + "\n")

run_grouped_insights_module(clean_df)

In [ ]:
def run_distribution_trends_module(clean_df):
    """Module 4: Tracks historical timelines and category volume distributions."""
    print("\n" + "="*20 + " MODULE 4: DISTRIBUTION & TRENDS " + "="*20)
    
    # Production Safeguard: Filter out IMDb-only records
    netflix_only = clean_df[clean_df["show_id"].notnull()]
    
    # Rating trend by release year [ Expected: "release_year", "Avg_Rating" ]
    print("IMDb Rating Trend by Release Year (Top 10 Sample Rows):")
    trend_by_release_year = (
        netflix_only.groupby("release_year")["Rating"]
        .mean()
        .reset_index()
        .sort_values(by="release_year")
        .head(10)
        .rename(columns={"Rating": "Avg_Rating"})
    )
    print(trend_by_release_year.to_string(index=False))
    print("-" * 73)
    
    # Count of Movies vs TV Shows with IMDb Rating > 8 [ Expected: "type", "Count" ]
    print("Count of Highly-Rated Content (Rating > 8) by Type:")
    tv_and_movies_with_rating_more_than_8 = (
        netflix_only.loc[
            netflix_only["Rating"] > 8
        ]
        .groupby("type")
        .size()
        .reset_index(name="Count")
    )
    print(tv_and_movies_with_rating_more_than_8.to_string(index=False))
    
    print("="*73 + "\n")

run_distribution_trends_module(clean_df)

In [ ]:
def run_analytics_netflix_pipeline(netflix_path, rating_path):
    """The Master Orchestrator: Executes the entire ETL and reporting pipeline."""
    print("=== INITIALIZING PRODUCTION ANALYTICS PIPELINE ===")

    # Pipeline entry point, Extract/load & clean data
    clean_df = load_and_clean_data(netflix_path, rating_path)

    # Execute business intelligence modules sequentially
    run_quality_and_coverage_module(clean_df)
    run_top_performers_module(clean_df)
    run_grouped_insights_module(clean_df)
    run_distribution_trends_module(clean_df)

    print("Pipeline execution successful, all modules rendered...")

run_analytics_netflix_pipeline("netflix_titles.csv", "imdb_ratings.csv")